# 02 — Build Silver source tables

AIDP operational pipeline notebook. It uses Volumes and governed tables only; the Git checkout is code only.

In [ ]:
from pathlib import Path
import os
import sys
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
root = Path.cwd().resolve()
for candidate in (Path(os.environ['AIDP_PROJECT_ROOT']).resolve(), root, *root.parents) if os.getenv('AIDP_PROJECT_ROOT') else (root, *root.parents):
    if (candidate / 'src').is_dir() and (candidate / 'config/settings.yaml').is_file():
        root = candidate
        break
else:
    raise RuntimeError('Project code root not found. Clone the repository; do not use it as a data store.')
sys.path.insert(0, str(root / 'src'))
from aidp_pipeline import LANDING_COUNTS, RAW_SESSIONS, STRUCTURAL_KEYS, landing_paths, load_aidp_config, missing_landing_paths, normalize_aidp_catalog_types, resolve_table_location, table_name
config = load_aidp_config(root / 'config/settings.yaml')


In [ ]:
from pathlib import Path

p = Path("/Workspace/redbull-race-intelligence-v2/src/aidp_pipeline.py")

print("exists:", p.exists())
print(
    "normalizer exists in file:",
    "def normalize_aidp_catalog_types" in p.read_text()
)

In [ ]:
import sys
import importlib

sys.modules.pop("aidp_pipeline", None)
importlib.invalidate_caches()

import aidp_pipeline

print(aidp_pipeline.__file__)
print(
    "normalizer available:",
    hasattr(aidp_pipeline, "normalize_aidp_catalog_types")
)

In [ ]:
from aidp_pipeline import normalize_aidp_catalog_types

print("IMPORT OK")

In [ ]:
missing = missing_landing_paths(config, lambda path: Path(path).is_file())
if missing:
    raise RuntimeError('Structured Bronze landing is missing. Run Notebook 01 and promote the locally validated Parquet artifacts first.')
for dataset, expected in LANDING_COUNTS.items():
    source_path = landing_paths(config)[dataset]
    frame = spark.read.parquet(source_path)
    for key in STRUCTURAL_KEYS[dataset]:
        if key not in frame.columns:
            raise ValueError(f'{dataset} is missing structural key {key}')
    if frame.count() != expected or frame.filter(' OR '.join(f'`{key}` IS NULL' for key in STRUCTURAL_KEYS[dataset])).count():
        raise ValueError(f'{dataset} did not satisfy validated landing checks')
    normalized = frame.withColumn('_aidp_bronze_source_path', F.lit(source_path))
    normalized = normalize_aidp_catalog_types(normalized)
    target = table_name(config, 'silver_schema', dataset)
    existing_location = resolve_table_location(spark, target)
    try:
        spark.sql(f'DROP TABLE IF EXISTS {target}')
        (normalized.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(target))
    except Exception as exc:
        location = existing_location or resolve_table_location(spark, target) or 'unavailable'
        raise RuntimeError(f'Unable to recreate Silver table {target}. Existing/resolved Delta storage location: {location}. Remove stale POC Delta storage only under approved governance, then retry.') from exc
    published = spark.table(target)
    actual = published.count()
    if actual != expected:
        raise ValueError(f'{target} row count mismatch after write: expected {expected}, got {actual}')
    published.printSchema()
    timestamp_ntz = [field.name for field in published.schema.fields if type(field.dataType).__name__ == 'TimestampNTZType']
    if timestamp_ntz:
        raise ValueError(f'{target} still contains TimestampNTZType columns: {timestamp_ntz}')
    print(f'Recreated historical Silver: {target}; rows={actual}; TimestampNTZType=none')
